<a href="https://colab.research.google.com/github/Mufliha-Hafsyah/Machine-Learning/blob/main/Week06/Praktikum3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **PRAKTIKUM PEMBELAJARAN MESIN**

**Nama**  : Mufliha Hafsyah Shahieza <br>
**NIM**   : 244107020147 <br>
**Kelas** : TI-3G

---

# **JS06 - ANN (Approximate Nearest Neighbors)**<hr>
# **Praktikum 3**<hr>

In [ ]:
!pip install hnswlib

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for hnswlib: filename=hnswlib-0.8.0-cp313-cp313-linux_x86_64.whl size=2679797 sha256=efa78f2987259a58c8f8811f2928f3659839d4b6c8d3f7bd15c0274e582fa5b2
  Stored in directory: /root/.cache/pip/wheels/35/04/88/b31765a4b9957705e18065db4657e61fc8da54f50e3ef0b67e
Successfully built hnswlib


In [ ]:
import hnswlib
import numpy as np
import time
from sklearn.neighbors import NearestNeighbors

def percobaan(n_data, dim, metric, k=5, seed=42):
    np.random.seed(seed)
    data = np.random.random((n_data, dim)).astype(np.float32)
    query = np.full((1, dim), 0.5, dtype=np.float32)

    # Exact NN (brute force)
    sk_metric = 'euclidean' if metric == 'l2' else 'cosine'
    nn = NearestNeighbors(n_neighbors=k, algorithm='brute', metric=sk_metric)
    nn.fit(data)
    t = time.time()
    _, I1 = nn.kneighbors(query)
    t_enn = (time.time() - t) * 1000

    # HNSW
    p = hnswlib.Index(space=metric, dim=dim)
    p.init_index(max_elements=n_data, ef_construction=100, M=16)
    tb = time.time()
    p.add_items(data)
    t_build = time.time() - tb
    p.set_ef(50)
    t = time.time()
    I2, _ = p.knn_query(query, k=k)
    t_hnsw = (time.time() - t) * 1000

    cocok = len(set(I1[0].tolist()) & set(I2[0].tolist()))
    print(f"{metric} | {dim}D | n={n_data:,}")
    print("  Index ENN  :", I1[0].tolist())
    print("  Index HNSW :", I2[0].tolist(), f"(cocok {cocok}/{k})")
    print(f"  Waktu      : ENN={t_enn:.4f} ms, HNSW={t_hnsw:.4f} ms")
    print(f"  (waktu build HNSW: {t_build:.1f} detik)\n")

In [ ]:
for n in (1000, 1_000_000):
    for dim in (2, 5):
        for metric in ('l2', 'cosine'):
            percobaan(n, dim, metric)

l2 | 2D | n=1,000
  Index ENN  : [137, 170, 750, 316, 991]
  Index HNSW : [137, 170, 750, 316, 991] (cocok 5/5)
  Waktu      : ENN=8.2183 ms, HNSW=0.1123 ms
  (waktu build HNSW: 0.1 detik)

cosine | 2D | n=1,000
  Index ENN  : [2, 135, 413, 210, 700]
  Index HNSW : [2, 135, 413, 210, 700] (cocok 5/5)
  Waktu      : ENN=22.1603 ms, HNSW=0.0799 ms
  (waktu build HNSW: 0.1 detik)

l2 | 5D | n=1,000
  Index ENN  : [895, 451, 375, 386, 411]
  Index HNSW : [895, 451, 375, 386, 411] (cocok 5/5)
  Waktu      : ENN=7.1676 ms, HNSW=0.1309 ms
  (waktu build HNSW: 0.1 detik)

cosine | 5D | n=1,000
  Index ENN  : [248, 386, 229, 230, 437]
  Index HNSW : [248, 386, 229, 230, 437] (cocok 5/5)
  Waktu      : ENN=4.3950 ms, HNSW=0.1206 ms
  (waktu build HNSW: 0.1 detik)

l2 | 2D | n=1,000,000
  Index ENN  : [286942, 583487, 695123, 948064, 829288]
  Index HNSW : [286942, 583487, 695123, 948064, 829288] (cocok 5/5)
  Waktu      : ENN=73.1621 ms, HNSW=0.3414 ms
  (waktu build HNSW: 55.0 detik)

cosine | 

### **Hasil Percobaan 3: Exact NN (ENN) vs HNSW**

| Distance Metrics | Dimensi | Jumlah data | Hasil Index terdekat ENN vs HNSW | Waktu komputasi <br> ENN vs HNSW (ms) |
|---|---|---|---|---|
| L2 (Euclidean) | 2D | 1.000 | [137, 170, 750, 316, 991] vs [137, 170, 750, 316, 991] | 8.2183 vs 0.1123 |
| Cosine | 2D | 1.000 | [2, 135, 413, 210, 700] vs [2, 135, 413, 210, 700] | 22.1603 vs 0.0799 |
| L2 (Euclidean) | 5D | 1.000 | [895, 451, 375, 386, 411] vs [895, 451, 375, 386, 411] | 7.1676 vs 0.1309 |
| Cosine | 5D | 1.000 | [248, 386, 229, 230, 437] vs [248, 386, 229, 230, 437] | 4.3950 vs 0.1206 |
| L2 (Euclidean) | 2D | 1.000.000 | [286942, 583487, 695123, 948064, 829288] vs [286942, 583487, 695123, 948064, 829288] | 73.1621 vs 0.3414 |
| Cosine | 2D | 1.000.000 | [469917, 383546, 49557, 317128, 87009] vs [87009, 724780, 770704, 951783, 984390] | 32.2433 vs 0.1736 |
| L2 (Euclidean) | 5D | 1.000.000 | [794791, 501340, 912100, 84056, 660781] vs [794791, 501340, 912100, 84056, 660781] | 31.0869 vs 0.1826 |
| Cosine | 5D | 1.000.000 | [422252, 464287, 605535, 319582, 869360] vs [422252, 464287, 605535, 319582, 869360] | 95.3860 vs 0.1450 |

**Waktu build indeks HNSW (tidak termasuk waktu pencarian):**

| Jumlah data | Waktu build |
|---|---|
| 1.000 | sekitar 0,1 detik |
| 1.000.000 | sekitar 55 sampai 100 detik |

**Analisis:**

1. **Akurasi.** HNSW menghasilkan tetangga yang sama persis dengan Exact NN (cocok 5/5) pada 7 dari 8 percobaan, termasuk pada data 5D dengan 1.000.000 data. Satu-satunya pengecualian adalah Cosine 2D dengan 1.000.000 data (cocok 1/5). Pada kasus ini selisih sudut antar tetangga terdekat sangat kecil, lebih kecil dari presisi `float32`, sehingga urutan "terdekat" menjadi tidak stabil, bahkan pada Exact NN sendiri.
2. **Kecepatan.** Pada 1.000.000 data, pencarian HNSW sekitar 170 sampai 660 kali lebih cepat dari Exact NN. Pada 1.000 data HNSW juga lebih cepat, tetapi waktu Exact NN di sini ikut terdongkrak overhead pemanggilan `NearestNeighbors` (scikit-learn), sehingga selisihnya tidak sepenuhnya mencerminkan perbedaan algoritma.
3. **Pengaruh dimensi.** Berbeda dengan IVF+PQ pada Praktikum 2, akurasi HNSW tidak turun dari 2D ke 5D (L2 tetap 5/5), karena HNSW tidak mengompresi vektor, hanya menavigasi graf tetangga.
4. **Biaya build.** Keunggulan pencarian HNSW dibayar di waktu pembangunan indeks, yaitu sekitar 0,1 detik untuk 1.000 data dibanding 55 sampai 100 detik untuk 1.000.000 data. Karena itu HNSW cocok untuk data yang indeksnya dibangun sekali dan dipakai untuk banyak pencarian.
5. **Pengaruh metrik.** Hasil L2 dan Cosine berbeda karena L2 mengukur jarak absolut sedangkan Cosine hanya mengukur sudut antar vektor.